# 🤖 Notebook 03 – Model Training Walkthrough
Step-by-step reproduction of the SafeGuard accident risk model, with metrics visualisations, confusion matrix, ROC curve, and feature importance.

In [ ]:
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

sys.path.insert(0, str(Path('data-science/src').resolve()))
from features import FEATURES, _haversine_km, encode_weather, encode_traffic, encode_road_type, encode_lighting

RANDOM_STATE = 42

# Load data
accidents = pd.read_csv('data/processed/accidents_clean.csv')
hotspots  = pd.read_csv('data/processed/hotspots.csv')
print(f'Accidents: {len(accidents):,}  |  Hotspots: {len(hotspots):,}')

## 1. Build Target Variable
`risk = 1` if severity is **severe/fatal** OR the accident is within **500 m** of a HIGH/MODERATE hotspot.

In [ ]:
# Severity-based risk
severity_risk = accidents['severity'].isin(['severe', 'fatal'])

# Proximity-based risk
hs_risk = hotspots[hotspots['risk_level'].isin(['HIGH', 'MODERATE'])]
if hs_risk.empty:
    proximity_risk = pd.Series(False, index=accidents.index)
else:
    hs_lats, hs_lons = hs_risk['latitude'].values, hs_risk['longitude'].values
    def _near(row):
        dists = _haversine_km(row['latitude'], row['longitude'], hs_lats, hs_lons)
        return bool(np.min(dists) <= 0.5)
    proximity_risk = accidents.apply(_near, axis=1)

y = (severity_risk | proximity_risk).astype(int)
print(f'Risk=0: {(y==0).sum():,}  |  Risk=1: {(y==1).sum():,}')

## 2. Feature Engineering
Encode categoricals, compute speed ratio, distance to hotspot, and local accident density.

In [ ]:
df = accidents.copy()
df['weather_code']   = df['weather'].map(encode_weather)
df['traffic_code']   = df['traffic_density'].map(encode_traffic)
df['road_type_code'] = df['road_type'].map(encode_road_type)
df['lighting_code']  = df['lighting'].map(encode_lighting)
df['speed_ratio']    = df['speed_kmh'] / df['speed_limit_kmh'].clip(lower=1)

# Distance to nearest hotspot
hs_lats = hotspots['latitude'].values
hs_lons = hotspots['longitude'].values
hs_sev  = hotspots['severity_index'].values
dist_mat = np.stack([
    _haversine_km(lat, lon, hs_lats, hs_lons)
    for lat, lon in zip(df['latitude'].values, df['longitude'].values)
])
nearest_idx = np.argmin(dist_mat, axis=1)
df['distance_to_hotspot_km'] = dist_mat[np.arange(len(df)), nearest_idx]
df['severity_index']         = hs_sev[nearest_idx]

# Historical accident count within 1 km
acc_lats, acc_lons = df['latitude'].values, df['longitude'].values
dist_self = np.stack([
    _haversine_km(lat, lon, acc_lats, acc_lons)
    for lat, lon in zip(acc_lats, acc_lons)
])
df['historical_accident_count'] = (dist_self <= 1.0).sum(axis=1) - 1
df = df.rename(columns={'speed_kmh': 'current_speed_kmh'})

X = df[FEATURES]
print(f'Feature matrix: {X.shape}')
X.head()

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 4. Train Models

In [ ]:
# Logistic Regression (baseline)
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
lr_pipe.fit(X_train, y_train)

# Random Forest (main model)
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(
        n_estimators=200, max_depth=15,
        random_state=RANDOM_STATE, n_jobs=-1
    )),
])
rf_pipe.fit(X_train, y_train)
print('Training complete.')

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, pipe) in zip(axes, [('Logistic Regression', lr_pipe), ('Random Forest', rf_pipe)]):
    y_pred = pipe.predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low Risk','High Risk'],
                yticklabels=['Low Risk','High Risk'])
    f1 = f1_score(y_test, y_pred, zero_division=0)
    ax.set_title(f'{name}\nF1={f1:.4f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 6. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, pipe, color in [
    ('Logistic Regression', lr_pipe, 'royalblue'),
    ('Random Forest',       rf_pipe,  'crimson'),
]:
    y_prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=color, linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
importances = rf_pipe['clf'].feature_importances_
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': importances})\
            .sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(feat_df['feature'], feat_df['importance'], color='steelblue', edgecolor='white')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Random Forest – Feature Importance')
plt.tight_layout()
plt.show()

## 8. Summary Metrics

In [ ]:
rows = []
for name, pipe in [('Logistic Regression', lr_pipe), ('Random Forest', rf_pipe)]:
    yp   = pipe.predict(X_test)
    yprb = pipe.predict_proba(X_test)[:, 1]
    rows.append({
        'Model':     name,
        'Precision': round(precision_score(y_test, yp, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, yp, zero_division=0), 4),
        'F1':        round(f1_score(y_test, yp, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, yprb), 4),
    })
pd.DataFrame(rows).set_index('Model')

---
*End of Model Training notebook.*